# Diagnóstico: DB, tamaños de corrida y viabilidad del cache
Objetivo: medir cuánto pesa cargar una corrida, cuántos pasos/agentes tiene,
y si el cache en `dcc.Store` de Dash es viable (límite ~10 MB por Store).

**Ejecutar desde la raíz del proyecto** con el environment del proyecto activado.

In [3]:
import sys, time, json, sqlite3
from pathlib import Path
import pandas as pd
import numpy as np

# ── Ajusta esta ruta si tu carpeta de resultados está en otro lugar ────────────
DATA_DIR = Path("../resultados/")
DB_PATH  = DATA_DIR / "dunas.db"
SUM_PATH = DATA_DIR / "summary.parquet"

print(f"DB     : {DB_PATH} — existe: {DB_PATH.exists()}")
print(f"Summary: {SUM_PATH} — existe: {SUM_PATH.exists()}")

DB     : ..\resultados\dunas.db — existe: True
Summary: ..\resultados\summary.parquet — existe: True


## 1 · Columnas del summary y primeras filas

In [4]:
summary = pd.read_parquet(SUM_PATH)
print(f"Filas: {len(summary)}  |  Columnas: {list(summary.columns)}")
summary.head(5)

Filas: 162  |  Columnas: ['run_id', 'wind_regime', 'qsat', 'q0ratio', 'qshift_ratio', 'lambda2_mean', 'lambda2_std', 'outflux_mode', 'dt', 'n_steps_run', 'n_dunes_final', 'mean_width_final', 'std_width_final', 'mean_asymmetry_final', 'calving_count', 'collision_count', 'merging_count', 'exchange_count', 'fragmentation_count', 'calving_rate', 'seed']


,run_id,wind_regime,qsat,q0ratio,qshift_ratio,lambda2_mean,lambda2_std,outflux_mode,dt,n_steps_run,...,mean_width_final,std_width_final,mean_asymmetry_final,calving_count,collision_count,merging_count,exchange_count,fragmentation_count,calving_rate,seed
0,run_9a22a245,unimodal,30.0,0.0,0.0,1.8,0.00,Duran,0.125,1415,...,0.0,0.0,0.0,0,0,0,0,0,0.0,980330
1,run_36ec2755,unimodal,30.0,0.0,0.0,1.8,0.00,Hersen,0.125,1415,...,0.0,0.0,0.0,0,0,0,0,0,0.0,980330
2,run_2dfd3041,unimodal,30.0,0.0,0.0,1.8,0.25,Hersen,0.125,1415,...,0.0,0.0,0.0,0,0,0,0,0,0.0,980330
3,run_d7df0708,unimodal,30.0,0.0,0.0,1.8,0.25,Duran,0.125,1415,...,0.0,0.0,0.0,0,0,0,0,0,0.0,980330
4,run_ebb42c23,unimodal,30.0,0.0,0.0,1.8,0.50,Duran,0.125,1415,...,0.0,0.0,0.0,0,0,0,0,0,0.0,980330


## 2 · Columnas de colisiones presentes

In [5]:
collision_cols = ["calving_count", "collision_count",
                  "merging_count", "exchange_count", "fragmentation_count",
                  "calving_rate"]
present = [c for c in collision_cols if c in summary.columns]
missing = [c for c in collision_cols if c not in summary.columns]
print(f"Presentes : {present}")
print(f"Ausentes  : {missing}")
if present:
    display(summary[present].describe())

Presentes : ['calving_count', 'collision_count', 'merging_count', 'exchange_count', 'fragmentation_count', 'calving_rate']
Ausentes  : []


,calving_count,collision_count,merging_count,exchange_count,fragmentation_count,calving_rate
count,162.000000,162.000000,162.000000,162.000000,162.000000,162.000000
mean,637.333333,620.635802,341.345679,186.290123,93.000000,0.318667
std,1290.906798,1471.826278,854.635292,429.913738,254.627288,0.645453
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,93.000000,19.500000,10.000000,2.000000,0.000000,0.046500
75%,660.000000,360.000000,256.000000,116.000000,16.500000,0.330000
max,7523.000000,9187.000000,5207.000000,2544.000000,1438.000000,3.761500


## 3 · Distribución de n_steps_run y n_dunes_final

In [6]:
for col in ["n_steps_run", "n_dunes_final"]:
    if col in summary.columns:
        s = summary[col].describe()
        print(f"{col}:  min={s['min']:.0f}  p50={s['50%']:.0f}  max={s['max']:.0f}")

n_steps_run:  min=354  p50=2000  max=2000
n_dunes_final:  min=0  p50=10  max=455


## 4 · Tamaño real de agent_data en la DB por corrida
Muestreamos 5 corridas: la más corta, la más larga, y 3 aleatorias.

In [7]:
con = sqlite3.connect(DB_PATH, timeout=30)

# Contar filas en agent_data por corrida
agent_counts = pd.read_sql(
    "SELECT run_id, COUNT(*) as n_rows FROM agent_data GROUP BY run_id ORDER BY n_rows",
    con
)
print(f"Corridas con agent_data: {len(agent_counts)}")
print(f"Filas de agent_data:  min={agent_counts.n_rows.min()}  "
      f"p50={agent_counts.n_rows.median():.0f}  max={agent_counts.n_rows.max()}")
print()
display(agent_counts.describe())
con.close()

Corridas con agent_data: 108
Filas de agent_data:  min=5482  p50=113584  max=729119



,n_rows
count,108.000000
mean,206430.222222
std,218259.779937
min,5482.000000
25%,27365.750000
50%,113584.000000
75%,333226.000000
max,729119.000000


## 5 · Tiempo de carga y tamaño en memoria de corridas representativas

In [8]:
con = sqlite3.connect(DB_PATH, timeout=30)
agent_counts_df = pd.read_sql(
    "SELECT run_id, COUNT(*) as n_rows FROM agent_data GROUP BY run_id ORDER BY n_rows",
    con
)
con.close()

if len(agent_counts_df) == 0:
    print("No hay agent_data en la DB.")
else:
    # Elegir corridas representativas
    n = len(agent_counts_df)
    idx_sample = sorted(set([
        0,                      # más corta
        n // 4,
        n // 2,                 # mediana
        3 * n // 4,
        n - 1,                  # más larga
    ]))
    sample_ids = agent_counts_df.iloc[idx_sample]["run_id"].tolist()

    # Agregar run_storage al path si hace falta
    if "scripts" not in sys.path:
        sys.path.insert(0, str(Path(".")))

    from scripts.run_storage import RunStorage

    results = []
    for rid in sample_ids:
        t0 = time.perf_counter()
        run_data = RunStorage.load_run(DATA_DIR, rid)
        t_load = time.perf_counter() - t0

        agents_df = run_data["agents"]
        model_df  = run_data["model"]

        # Tamaño en memoria del DataFrame de agentes
        mem_agents = agents_df.memory_usage(deep=True).sum() / 1024**2  # MB
        mem_model  = model_df.memory_usage(deep=True).sum()  / 1024**2

        # Número de pasos y agentes
        if isinstance(agents_df.index, pd.MultiIndex):
            steps = agents_df.index.get_level_values(0).unique()
            n_steps = len(steps)
            # agentes en el último paso
            n_agents_last = len(agents_df.xs(steps[-1], level=0))
        else:
            n_steps = 0
            n_agents_last = 0

        # Tamaño JSON serializado (lo que iría al dcc.Store)
        t1 = time.perf_counter()
        agents_json = agents_df.reset_index().to_json(orient="records")
        model_json  = model_df.reset_index().to_json(orient="records")
        t_ser = time.perf_counter() - t1
        size_json_mb = (len(agents_json) + len(model_json)) / 1024**2

        results.append({
            "run_id":         rid,
            "n_steps":        n_steps,
            "n_agents_final": n_agents_last,
            "n_agent_rows":   len(agents_df),
            "t_load_s":       round(t_load, 3),
            "mem_agents_MB":  round(mem_agents, 2),
            "mem_model_MB":   round(mem_model, 2),
            "json_total_MB":  round(size_json_mb, 2),
            "t_serialize_s":  round(t_ser, 3),
        })

    diag = pd.DataFrame(results)
    display(diag)
    print()
    print(f"Límite dcc.Store Dash: ~10 MB por store")
    print(f"¿Viable cache en Store? ",
          "✅ SÍ" if diag["json_total_MB"].max() < 8 else "⚠️  EXCEDE límite — necesita compresión o cache alternativo")

,run_id,n_steps,n_agents_final,n_agent_rows,t_load_s,mem_agents_MB,mem_model_MB,json_total_MB,t_serialize_s
0,run_8d5e7cde,1925,4,5482,0.055,0.45,0.26,1.93,0.012
1,run_5e31bba4,2000,18,34531,0.216,2.85,0.26,8.32,0.050
2,run_134d6202,1991,61,114393,0.666,9.08,0.26,24.99,0.148
3,run_024762ac,2000,189,333253,2.197,27.63,0.26,65.72,0.404
4,run_c9605835,2000,451,729119,4.609,59.97,0.26,155.75,0.925



Límite dcc.Store Dash: ~10 MB por store
¿Viable cache en Store?  ⚠️  EXCEDE límite — necesita compresión o cache alternativo


## 6 · Costo de agents_at_step vs carga completa
Simula lo que hace cb_step en cada tick del slider: carga todo el agent_data
y luego filtra un solo paso. Medimos el overhead.

In [9]:
if len(agent_counts_df) > 0:
    # Usar la corrida mediana
    rid_median = agent_counts_df.iloc[len(agent_counts_df) // 2]["run_id"]
    run_data = RunStorage.load_run(DATA_DIR, rid_median)
    agents_df = run_data["agents"]

    if isinstance(agents_df.index, pd.MultiIndex):
        all_steps = sorted(agents_df.index.get_level_values(0).unique().tolist())
        test_steps = all_steps[::max(1, len(all_steps) // 5)]  # 5 pasos de muestra

        times_full = []
        for step in test_steps:
            t0 = time.perf_counter()
            _  = RunStorage.load_run(DATA_DIR, rid_median)  # carga completa
            times_full.append(time.perf_counter() - t0)

        times_slice = []
        for step in test_steps:
            t0 = time.perf_counter()
            _  = agents_df.xs(step, level=0)  # solo slice (ya cargado)
            times_slice.append(time.perf_counter() - t0)

        print(f"Corrida: {rid_median}  |  pasos totales: {len(all_steps)}")
        print(f"Carga completa (por tick actual):  {np.mean(times_full)*1000:.1f} ms ± {np.std(times_full)*1000:.1f}")
        print(f"Solo slice (si data ya en cache):  {np.mean(times_slice)*1000:.3f} ms ± {np.std(times_slice)*1000:.3f}")
        print(f"Ganancia: {np.mean(times_full)/max(np.mean(times_slice),1e-6):.0f}×")
    else:
        print("agent_data no tiene MultiIndex Step/AgentID — revisar estructura.")

Corrida: run_134d6202  |  pasos totales: 1991
Carga completa (por tick actual):  643.0 ms ± 33.3
Solo slice (si data ya en cache):  0.319 ms ± 0.432
Ganancia: 2018×


## 7 · Estructura del agent_data (columnas disponibles)

In [10]:
if len(agent_counts_df) > 0:
    rid_any = agent_counts_df.iloc[len(agent_counts_df) // 2]["run_id"]
    run_data = RunStorage.load_run(DATA_DIR, rid_any)
    agents_df = run_data["agents"]
    model_df  = run_data["model"]

    print("=== agent_data ===")
    print(f"Shape: {agents_df.shape}")
    print(f"Index: {agents_df.index.names}")
    print(f"Columnas: {list(agents_df.columns)}")
    print()

    # ¿Hay columnas de tipo de colisión a nivel de agente?
    calving_agent_cols = [c for c in agents_df.columns
                          if any(k in c.lower() for k in ["calv", "merg", "exch", "frag", "collide"])]
    print(f"Columnas de eventos en agent_data: {calving_agent_cols or 'ninguna'}")

    print()
    print("=== model_data ===")
    print(f"Shape: {model_df.shape}")
    print(f"Columnas: {list(model_df.columns)}")

    calving_model_cols = [c for c in model_df.columns
                          if any(k in c.lower() for k in ["calv", "merg", "exch", "frag", "collide"])]
    print(f"Columnas de eventos en model_data: {calving_model_cols or 'ninguna'}")

    # Muestra primeras filas
    display(agents_df.head(3))
    display(model_df.head(3))

=== agent_data ===
Shape: (114393, 9)
Index: ['Step', 'AgentID']
Columnas: ['lw', 'rw', 'width', 'asymmetry', 'asym_ratio', 'lambda2', 'morphotype', 'pos_x', 'pos_y']

Columnas de eventos en agent_data: ninguna

=== model_data ===
Shape: (2000, 16)
Columnas: ['step', 'N_dunes', 'mean_width', 'std_width', 'mean_asymmetry', 'calvings_this_step', 'collisions_this_step', 'merging_this_step', 'exchange_this_step', 'fragmentation_this_step', 'calving_count', 'collision_count', 'merging_count', 'exchange_count', 'fragmentation_count', 'wind_angle_deg']
Columnas de eventos en model_data: ['calvings_this_step', 'merging_this_step', 'exchange_this_step', 'fragmentation_this_step', 'calving_count', 'merging_count', 'exchange_count', 'fragmentation_count']


,,lw,rw,width,asymmetry,asym_ratio,lambda2,morphotype,pos_x,pos_y
Step,AgentID,,,,,,,,,
10,1,11.500000,11.500000,23.000000,0.000000,1.000000,1.8,barchan,4727.113200,9999.999982
11,1,11.500594,11.499403,22.999998,0.000052,1.000104,1.8,barchan,4727.122536,9997.700001
12,1,11.498987,11.500993,22.999980,0.000087,0.999826,1.8,barchan,4727.097638,9995.400136


,step,N_dunes,mean_width,std_width,mean_asymmetry,calvings_this_step,collisions_this_step,merging_this_step,exchange_this_step,fragmentation_this_step,calving_count,collision_count,merging_count,exchange_count,fragmentation_count,wind_angle_deg
Step,,,,,,,,,,,,,,,,
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,270.483981
1,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,267.507572
2,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,266.137354


## 8 · Veredicto final

Basado en las celdas anteriores:
- Si `json_total_MB` ≤ 8 MB → **cache en dcc.Store viable** (sin compresión).
- Si `json_total_MB` > 8 MB → necesitamos cache en disco (archivo temporal por session) o comprimir con zlib+base64 dentro del Store.
- Las columnas en `model_data` determinan si el filtro de tipo de calveo es posible en la serie de tiempo, o solo en el summary.

In [11]:
print("Resumen ejecutivo")
print("=" * 50)
if 'diag' in dir():
    print(f"Max json serializado:  {diag['json_total_MB'].max():.2f} MB")
    print(f"Max t_load:            {diag['t_load_s'].max():.3f} s")
    print(f"Max n_steps:           {diag['n_steps'].max()}")
    print(f"Max n_agent_rows:      {diag['n_agent_rows'].max()}")
    ok = diag['json_total_MB'].max() < 8
    print()
    print("Estrategia de cache recomendada:")
    if ok:
        print("  → dcc.Store con JSON directo (sin compresión)")
        print("  → agents como list of records, model como list of records")
        print("  → cb_load_run serializa 1 vez; cb_step solo hace pd.DataFrame + xs()")
    else:
        print("  → cache en disco: archivo .pkl.gz por run_id en /tmp/")
        print("  → dcc.Store solo guarda run_id (string)")
        print("  → lru_cache(maxsize=3) sobre load_run para sesión única")

Resumen ejecutivo
Max json serializado:  155.75 MB
Max t_load:            4.609 s
Max n_steps:           2000
Max n_agent_rows:      729119

Estrategia de cache recomendada:
  → cache en disco: archivo .pkl.gz por run_id en /tmp/
  → dcc.Store solo guarda run_id (string)
  → lru_cache(maxsize=3) sobre load_run para sesión única


In [1]:
import pandas as pd
df = pd.read_parquet("../resultados/summary.parquet")
print(df.shape)
print(df.columns.tolist())
print(df.head(10).to_string())

(162, 21)
['run_id', 'wind_regime', 'qsat', 'q0ratio', 'qshift_ratio', 'lambda2_mean', 'lambda2_std', 'outflux_mode', 'dt', 'n_steps_run', 'n_dunes_final', 'mean_width_final', 'std_width_final', 'mean_asymmetry_final', 'calving_count', 'collision_count', 'merging_count', 'exchange_count', 'fragmentation_count', 'calving_rate', 'seed']
         run_id wind_regime  qsat  q0ratio  qshift_ratio  lambda2_mean  lambda2_std outflux_mode     dt  n_steps_run  n_dunes_final  mean_width_final  std_width_final  mean_asymmetry_final  calving_count  collision_count  merging_count  exchange_count  fragmentation_count  calving_rate    seed
0  run_9a22a245    unimodal  30.0      0.0           0.0           1.8         0.00        Duran  0.125         1415              0               0.0              0.0                   0.0              0                0              0               0                    0           0.0  980330
1  run_36ec2755    unimodal  30.0      0.0           0.0           1.8   

In [2]:
import sqlite3, pandas as pd
con = sqlite3.connect("../resultados/dunas.db")
agent = pd.read_sql("SELECT * FROM agent_data LIMIT 5", con)
model = pd.read_sql("SELECT * FROM model_data LIMIT 5", con)
print("agent_data columns:", agent.columns.tolist())
print("model_data columns:", model.columns.tolist())
print(agent.head(2).to_string())
print(model.head(2).to_string())
con.close()

agent_data columns: ['run_id', 'step', 'agent_id', 'data_json']
model_data columns: ['run_id', 'step', 'data_json']
         run_id  step  agent_id                                                                                                                                                                                                          data_json
0  run_a26424da     1         1                                               {"lw":46.0,"rw":46.0,"width":92.0,"asymmetry":0.0,"asym_ratio":1.0,"lambda2":1.4781128173,"morphotype":"transverse","pos_x":3544.4929239711,"pos_y":9999.9992872596}
1  run_a26424da     2         1  {"lw":44.0661745848,"rw":44.0708593005,"width":88.1370338853,"asymmetry":0.0000531526,"asym_ratio":0.9998937004,"lambda2":1.4781128173,"morphotype":"transverse","pos_x":3544.3093846567,"pos_y":9990.8011182396}
         run_id  step                                                                                                                                      

In [5]:
import sqlite3, pandas as pd, json

con = sqlite3.connect("../resultados/dunas.db")

# Expandir model_data completo
df = pd.read_sql("""
    SELECT m.run_id, m.step, m.data_json,
           r.lambda2_std, r.qshift_ratio, r.q0ratio,
           r.qsat, r.wind_regime, r.outflux_mode, r.seed
    FROM model_data m
    JOIN runs r ON m.run_id = r.run_id
""", con)
df_exp = pd.concat([
    df.drop(columns=['data_json']),
    pd.json_normalize(df['data_json'].apply(json.loads))
], axis=1)

# Expandir agent_data — solo último 20% de pasos por corrida
max_steps = pd.read_sql("SELECT run_id, MAX(step) as ms FROM agent_data GROUP BY run_id", con)
frames = []
for _, row in max_steps.iterrows():
    cutoff = int(row['ms'] * 0.8)
    q = f"""SELECT a.run_id, a.step, a.data_json,
                   r.lambda2_std, r.qshift_ratio, r.q0ratio,
                   r.qsat, r.wind_regime, r.outflux_mode
            FROM agent_data a JOIN runs r ON a.run_id = r.run_id
            WHERE a.run_id = '{row['run_id']}' AND a.step >= {cutoff}"""
    frames.append(pd.read_sql(q, con))
con.close()

agent_df = pd.concat(frames, ignore_index=True)
agent_exp = pd.concat([
    agent_df.drop(columns=['data_json']),
    pd.json_normalize(agent_df['data_json'].apply(json.loads))
], axis=1)

df_exp.to_parquet("../tmp/model_data_exp.parquet", index=False)
agent_exp.to_parquet("../tmp/agent_data_exp.parquet", index=False)
print("model_data shape:", df_exp.shape)
print("agent_data shape:", agent_exp.shape)
print("model cols:", df_exp.columns.tolist())
print("agent cols:", agent_exp.columns.tolist())

ValueError: Duplicate column names found: ['run_id', 'step', 'lambda2_std', 'qshift_ratio', 'q0ratio', 'qsat', 'wind_regime', 'outflux_mode', 'seed', 'step', 'N_dunes', 'mean_width', 'std_width', 'mean_asymmetry', 'calvings_this_step', 'collisions_this_step', 'merging_this_step', 'exchange_this_step', 'fragmentation_this_step', 'calving_count', 'collision_count', 'merging_count', 'exchange_count', 'fragmentation_count', 'wind_angle_deg']

In [7]:
import pandas as pd
df = pd.read_parquet("../resultados/summary.parquet")
df.to_csv("../summary.csv", index=False)
print("listo")

listo
